In [28]:
pip install  -U transformers

In [ ]:
# use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation",model="ibm-granite/granite-3.3-2b-instruct")
message = [
    {"role":"user","content": "who are you?"},
]
pipe(message)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# load model directly
from transformers import AutoTokenizer,AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
message = [
    {"role":"user","content": "who are you?"}
]
inputs = tokenizer.apply_chat_template(
  message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import gradio as gr
import re

tables = {
    "users": ["id","name","email","signup_date","age","country","status"],
    "orders": ["id","user_id","product_name","amount","order_date","status"],
    "products": ["id","name","price","category","stock"],
    "transactions": ["id","user_id","amount","date","type"]
}

def generate_sql(query):

    q = query.lower()
    table = "users"

    if "order" in q:
        table = "orders"
    elif "product" in q:
        table = "products"
    elif "transaction" in q:
        table = "transactions"

    columns = "*"

    if "name" in q:
        columns = "name"

    if "email" in q:
        columns = "email"

    sql = f"SELECT {columns} FROM {table}"

    if "last month" in q:
        sql += " WHERE date >= DATE('now','-1 month')"

    if "active" in q:
        sql += " WHERE status='active'"

    sql += " LIMIT 10"

    return f"""
✅ SQL Generated
"""

def chat(message, history):

    response = generate_sql(message)

    history.append((message,response))

    return history

with gr.Blocks(
    title="AI SQL Generator",
    css="""
    body{background:linear-gradient(135deg,#1a1a2e,#16213e);}
    """
) as demo:

    gr.Markdown("# 🤖 AI SQL Generator")

    chatbot = gr.Chatbot(height=450)

    txt = gr.Textbox(
        label="Ask Database",
        placeholder="Example: show active users last month"
    )

    txt.submit(chat,[txt,chatbot],chatbot)

demo.launch(share=True)

In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import sqlite3
import re
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse

# =========================
# DATABASE SCHEMA
# =========================

SCHEMA = {
    'users': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT',
        'email': 'TEXT',
        'signup_date': 'DATE',
        'age': 'INTEGER',
        'country': 'TEXT',
        'status': 'TEXT'
    },
    'orders': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'product_name': 'TEXT',
        'amount': 'REAL',
        'order_date': 'DATE',
        'status': 'TEXT'
    },
    'products': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT',
        'price': 'REAL',
        'category': 'TEXT',
        'stock': 'INTEGER'
    },
    'transactions': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'amount': 'REAL',
        'date': 'DATE',
        'type': 'TEXT'
    }
}

# =========================
# SECURITY & SQL GENERATION
# =========================

DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE', 'EXEC', 'EXECUTE']
DANGEROUS_CHARS = [';', '--', '/*', '*/', 'xp_', 'sp_']